
# Interactive Ball-Detection Parameter Tuning

This notebook is for **interactive tuning only**. It lets you:

- choose a frame from the **differential video**
- interactively adjust the detection parameters with sliders
- optionally restrict the allowed search region
- display the **top candidate guesses** overlaid on the frame
- inspect the binary threshold mask and candidate table

It is designed to help you find good parameters before running the full video export notebook.


In [1]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Polygon as MplPolygon
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout


In [11]:

# =========================
# USER INPUTS
# =========================

diff_video_path = "diff videos/adjacent_frame_difference.mp4"

# Set one of these:
use_scale_for_display = False
display_scale = 1.0  # only for plotting if you want smaller display in matplotlib

# Default starting parameters for the sliders
default_ball_diameter_px = 10.0
default_diff_thresh = 22
default_min_area_scale = 0.20
default_max_area_scale = 2.50
default_min_bbox_scale = 0.50
default_max_bbox_scale = 1.80
default_min_circularity = 0.35
default_min_mean_intensity = 15.0
default_max_aspect_ratio = 2.20
default_top_k = 5
default_gaussian_blur_ksize = 3  # must be odd; use 0 to disable

# Allowed-region mode:
# "none", "rectangle", or "polygon"
default_region_mode = "none"

# Rectangle defaults
default_rect_x0 = 0
default_rect_y0 = 0
default_rect_x1 = 400
default_rect_y1 = 400

# Polygon defaults: list of (x, y) points
default_polygon_points = [
    (100, 100),
    (500, 100),
    (550, 350),
    (80, 350),
]


In [15]:

def get_video_info(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        "width": width,
        "height": height,
        "fps": fps,
        "frame_count": frame_count,
    }

def read_video_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        raise ValueError(f"Could not read frame {frame_idx} from {video_path}")
    return frame

def to_gray(frame):
    if frame.ndim == 2:
        return frame
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

def maybe_blur(gray, ksize):
    if ksize is None or int(ksize) <= 1:
        return gray
    ksize = int(ksize)
    if ksize % 2 == 0:
        ksize += 1
    return cv2.GaussianBlur(gray, (ksize, ksize), 0)

def threshold_motion(gray, diff_thresh):
    _, binary = cv2.threshold(gray, int(diff_thresh), 255, cv2.THRESH_BINARY)
    return binary

def make_allowed_mask(shape_hw, mode="none", rect=None, polygon_points=None):
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)

    if mode == "none":
        mask[:, :] = 255
        return mask

    if mode == "rectangle":
        if rect is None:
            raise ValueError("rect must be provided for rectangle mode")
        x0, y0, x1, y1 = rect
        x0, x1 = sorted([int(x0), int(x1)])
        y0, y1 = sorted([int(y0), int(y1)])
        x0 = max(0, min(w - 1, x0))
        x1 = max(0, min(w, x1))
        y0 = max(0, min(h - 1, y0))
        y1 = max(0, min(h, y1))
        mask[y0:y1, x0:x1] = 255
        return mask

    if mode == "polygon":
        if polygon_points is None or len(polygon_points) < 3:
            raise ValueError("polygon_points must contain at least 3 points")
        pts = np.array(polygon_points, dtype=np.int32)
        cv2.fillPoly(mask, [pts], 255)
        return mask

    raise ValueError(f"Unknown mode: {mode}")

def circularity_from_stats(area, perimeter):
    if perimeter <= 0:
        return 0.0
    return float(4.0 * np.pi * area / (perimeter ** 2))

def extract_ball_candidates(
    diff_gray,
    ball_diameter_px,
    diff_thresh=22,
    min_area_scale=0.20,
    max_area_scale=2.50,
    min_bbox_scale=0.50,
    max_bbox_scale=1.80,
    min_circularity=0.35,
    min_mean_intensity=15.0,
    max_aspect_ratio=2.20,
    top_k=5,
    allowed_mask=None,
    gaussian_blur_ksize=3,
):
    diff_gray = to_gray(diff_gray)
    proc_gray = maybe_blur(diff_gray, gaussian_blur_ksize)
    binary = threshold_motion(proc_gray, diff_thresh)

    if allowed_mask is not None:
        binary = cv2.bitwise_and(binary, allowed_mask)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

    expected_area = np.pi * (ball_diameter_px / 2.0) ** 2
    min_area = min_area_scale * expected_area
    max_area = max_area_scale * expected_area
    min_bbox = min_bbox_scale * ball_diameter_px
    max_bbox = max_bbox_scale * ball_diameter_px

    candidates = []

    for label_id in range(1, num_labels):
        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        area = float(stats[label_id, cv2.CC_STAT_AREA])

        if area < min_area or area > max_area:
            continue
        if w < min_bbox or h < min_bbox:
            continue
        if w > max_bbox or h > max_bbox:
            continue

        aspect_ratio = max(w / max(h, 1), h / max(w, 1))
        if aspect_ratio > max_aspect_ratio:
            continue

        component_mask = (labels == label_id).astype(np.uint8) * 255
        contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        cnt = max(contours, key=cv2.contourArea)
        perimeter = float(cv2.arcLength(cnt, closed=True))
        circularity = circularity_from_stats(area, perimeter)
        if circularity < min_circularity:
            continue

        mean_intensity = float(cv2.mean(diff_gray, mask=component_mask)[0])
        if mean_intensity < min_mean_intensity:
            continue

        cx, cy = centroids[label_id]
        radius_est = 0.25 * (w + h)

        area_ratio = area / max(expected_area, 1e-6)
        area_score = 1.0 / (1.0 + abs(np.log(max(area_ratio, 1e-6))))
        circle_score = circularity
        intensity_score = mean_intensity / 255.0
        aspect_score = 1.0 / max(aspect_ratio, 1.0)

        score = (
            0 * area_score
            + 0.30 * circle_score
            + 0.60 * intensity_score
            + 0.10 * aspect_score
        )

        candidates.append({
            "label_id": int(label_id),
            "x": int(x),
            "y": int(y),
            "w": int(w),
            "h": int(h),
            "cx": float(cx),
            "cy": float(cy),
            "area": float(area),
            "mean_intensity": float(mean_intensity),
            "circularity": float(circularity),
            "aspect_ratio": float(aspect_ratio),
            "radius_est": float(radius_est),
            "score": float(score),
        })

    candidates = sorted(candidates, key=lambda c: c["score"], reverse=True)
    return candidates[:top_k], {
        "diff_gray": diff_gray,
        "proc_gray": proc_gray,
        "binary": binary,
        "labels": labels,
        "stats": stats,
        "centroids": centroids,
        "expected_area": expected_area,
        "num_labels": num_labels,
    }

def draw_candidates_on_frame(gray_frame, candidates, allowed_mask=None, show_labels=True):
    if gray_frame.ndim == 2:
        overlay = cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2BGR)
    else:
        overlay = gray_frame.copy()

    if allowed_mask is not None:
        boundary = cv2.Canny(allowed_mask, 50, 150)
        overlay[boundary > 0] = (255, 255, 0)

    for i, c in enumerate(candidates):
        cx, cy = int(round(c["cx"])), int(round(c["cy"]))
        r = max(2, int(round(c["radius_est"])))
        cv2.circle(overlay, (cx, cy), r, (0, 255, 0), 2)
        cv2.rectangle(overlay, (c["x"], c["y"]), (c["x"] + c["w"], c["y"] + c["h"]), (0, 128, 255), 1)
        if show_labels:
            txt = f"{i+1}: {c['score']:.2f}"
            cv2.putText(overlay, txt, (cx + 4, cy - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
    return overlay

def parse_polygon_text(text):
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    pts = []
    for ln in lines:
        parts = ln.replace(",", " ").split()
        if len(parts) != 2:
            continue
        x, y = float(parts[0]), float(parts[1])
        pts.append((int(round(x)), int(round(y))))
    return pts

def preview_detection(
    video_path,
    frame_idx,
    ball_diameter_px,
    diff_thresh,
    min_area_scale,
    max_area_scale,
    min_bbox_scale,
    max_bbox_scale,
    min_circularity,
    min_mean_intensity,
    max_aspect_ratio,
    top_k,
    gaussian_blur_ksize,
    region_mode,
    rect_x0,
    rect_y0,
    rect_x1,
    rect_y1,
    polygon_text,
    show_binary=True,
    show_processed=False,
):
    frame = read_video_frame(video_path, frame_idx)
    gray = to_gray(frame)

    allowed_mask = None
    polygon_points = None
    if region_mode == "rectangle":
        allowed_mask = make_allowed_mask(
            gray.shape,
            mode="rectangle",
            rect=(rect_x0, rect_y0, rect_x1, rect_y1),
        )
    elif region_mode == "polygon":
        polygon_points = parse_polygon_text(polygon_text)
        allowed_mask = make_allowed_mask(
            gray.shape,
            mode="polygon",
            polygon_points=polygon_points,
        )

    candidates, debug = extract_ball_candidates(
        diff_gray=gray,
        ball_diameter_px=ball_diameter_px,
        diff_thresh=diff_thresh,
        min_area_scale=min_area_scale,
        max_area_scale=max_area_scale,
        min_bbox_scale=min_bbox_scale,
        max_bbox_scale=max_bbox_scale,
        min_circularity=min_circularity,
        min_mean_intensity=min_mean_intensity,
        max_aspect_ratio=max_aspect_ratio,
        top_k=top_k,
        allowed_mask=allowed_mask,
        gaussian_blur_ksize=gaussian_blur_ksize,
    )

    overlay = draw_candidates_on_frame(gray, candidates, allowed_mask=allowed_mask, show_labels=True)

    cols = 3 if (show_binary or show_processed) else 1
    fig, axes = plt.subplots(1, cols, figsize=(6 * cols, 6))

    if cols == 1:
        axes = [axes]

    ax0 = axes[0]
    ax0.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax0.set_title(f"Overlay on frame {frame_idx}")
    ax0.axis("off")

    next_ax = 1
    if show_binary:
        axes[next_ax].imshow(debug["binary"], cmap="gray", vmin=0, vmax=255)
        axes[next_ax].set_title("Thresholded / filtered mask")
        axes[next_ax].axis("off")
        next_ax += 1

    if show_processed:
        axes[next_ax].imshow(debug["proc_gray"], cmap="gray", vmin=0, vmax=255)
        axes[next_ax].set_title("Processed grayscale")
        axes[next_ax].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Frame: {frame_idx}")
    print(f"Candidates returned: {len(candidates)}")
    print(f"Expected ball area: {debug['expected_area']:.2f} px^2")
    print()

    if len(candidates) == 0:
        print("No candidates found with current parameters.")
    else:
        for i, c in enumerate(candidates, start=1):
            print(
                f"{i}. center=({c['cx']:.1f}, {c['cy']:.1f}) "
                f"bbox=({c['w']}x{c['h']}) area={c['area']:.1f} "
                f"circ={c['circularity']:.3f} mean={c['mean_intensity']:.1f} "
                f"aspect={c['aspect_ratio']:.2f} score={c['score']:.3f}"
            )

    return candidates, debug, overlay


In [16]:

video_info = get_video_info(diff_video_path)
video_info


{'width': 1920, 'height': 1080, 'fps': 24.99, 'frame_count': 231}

In [17]:

frame_slider = widgets.IntSlider(
    value=min(0, max(video_info["frame_count"] - 1, 0)),
    min=0,
    max=max(video_info["frame_count"] - 1, 0),
    step=1,
    description="frame",
    continuous_update=False,
    layout=Layout(width="500px"),
)

ball_diameter_slider = widgets.FloatSlider(
    value=default_ball_diameter_px, min=2.0, max=40.0, step=0.5,
    description="ball px", continuous_update=False, layout=Layout(width="500px")
)

diff_thresh_slider = widgets.IntSlider(
    value=default_diff_thresh, min=1, max=255, step=1,
    description="thresh", continuous_update=False, layout=Layout(width="500px")
)

min_area_scale_slider = widgets.FloatSlider(
    value=default_min_area_scale, min=0.01, max=2.0, step=0.01,
    description="min area s", continuous_update=False, layout=Layout(width="500px")
)

max_area_scale_slider = widgets.FloatSlider(
    value=default_max_area_scale, min=0.10, max=8.0, step=0.05,
    description="max area s", continuous_update=False, layout=Layout(width="500px")
)

min_bbox_scale_slider = widgets.FloatSlider(
    value=default_min_bbox_scale, min=0.10, max=2.0, step=0.05,
    description="min bbox s", continuous_update=False, layout=Layout(width="500px")
)

max_bbox_scale_slider = widgets.FloatSlider(
    value=default_max_bbox_scale, min=0.20, max=5.0, step=0.05,
    description="max bbox s", continuous_update=False, layout=Layout(width="500px")
)

min_circularity_slider = widgets.FloatSlider(
    value=default_min_circularity, min=0.0, max=1.0, step=0.01,
    description="circularity", continuous_update=False, layout=Layout(width="500px")
)

min_mean_intensity_slider = widgets.FloatSlider(
    value=default_min_mean_intensity, min=0.0, max=255.0, step=1.0,
    description="min mean", continuous_update=False, layout=Layout(width="500px")
)

max_aspect_ratio_slider = widgets.FloatSlider(
    value=default_max_aspect_ratio, min=1.0, max=6.0, step=0.05,
    description="max aspect", continuous_update=False, layout=Layout(width="500px")
)

top_k_slider = widgets.IntSlider(
    value=default_top_k, min=1, max=20, step=1,
    description="top k", continuous_update=False, layout=Layout(width="500px")
)

gaussian_blur_slider = widgets.IntSlider(
    value=default_gaussian_blur_ksize, min=0, max=15, step=1,
    description="blur k", continuous_update=False, layout=Layout(width="500px")
)

region_mode_dropdown = widgets.Dropdown(
    options=["none", "rectangle", "polygon"],
    value=default_region_mode,
    description="region",
    layout=Layout(width="300px")
)

rect_x0_slider = widgets.IntSlider(
    value=default_rect_x0, min=0, max=max(video_info["width"] - 1, 0), step=1,
    description="x0", continuous_update=False, layout=Layout(width="300px")
)
rect_y0_slider = widgets.IntSlider(
    value=default_rect_y0, min=0, max=max(video_info["height"] - 1, 0), step=1,
    description="y0", continuous_update=False, layout=Layout(width="300px")
)
rect_x1_slider = widgets.IntSlider(
    value=min(default_rect_x1, video_info["width"]), min=0, max=max(video_info["width"], 1), step=1,
    description="x1", continuous_update=False, layout=Layout(width="300px")
)
rect_y1_slider = widgets.IntSlider(
    value=min(default_rect_y1, video_info["height"]), min=0, max=max(video_info["height"], 1), step=1,
    description="y1", continuous_update=False, layout=Layout(width="300px")
)

polygon_text = widgets.Textarea(
    value="\n".join([f"{x}, {y}" for x, y in default_polygon_points]),
    description="polygon",
    layout=Layout(width="350px", height="140px")
)

show_binary_checkbox = widgets.Checkbox(value=True, description="show binary mask")
show_processed_checkbox = widgets.Checkbox(value=False, description="show processed gray")

run_button = widgets.Button(description="Update preview", button_style="primary")
output = widgets.Output()

def _run_preview(_=None):
    with output:
        clear_output(wait=True)
        preview_detection(
            video_path=diff_video_path,
            frame_idx=frame_slider.value,
            ball_diameter_px=ball_diameter_slider.value,
            diff_thresh=diff_thresh_slider.value,
            min_area_scale=min_area_scale_slider.value,
            max_area_scale=max_area_scale_slider.value,
            min_bbox_scale=min_bbox_scale_slider.value,
            max_bbox_scale=max_bbox_scale_slider.value,
            min_circularity=min_circularity_slider.value,
            min_mean_intensity=min_mean_intensity_slider.value,
            max_aspect_ratio=max_aspect_ratio_slider.value,
            top_k=top_k_slider.value,
            gaussian_blur_ksize=gaussian_blur_slider.value,
            region_mode=region_mode_dropdown.value,
            rect_x0=rect_x0_slider.value,
            rect_y0=rect_y0_slider.value,
            rect_x1=rect_x1_slider.value,
            rect_y1=rect_y1_slider.value,
            polygon_text=polygon_text.value,
            show_binary=show_binary_checkbox.value,
            show_processed=show_processed_checkbox.value,
        )

run_button.on_click(_run_preview)

controls_left = VBox([
    frame_slider,
    ball_diameter_slider,
    diff_thresh_slider,
    min_area_scale_slider,
    max_area_scale_slider,
    min_bbox_scale_slider,
    max_bbox_scale_slider,
    min_circularity_slider,
    min_mean_intensity_slider,
    max_aspect_ratio_slider,
    top_k_slider,
    gaussian_blur_slider,
])

controls_right = VBox([
    region_mode_dropdown,
    HBox([rect_x0_slider, rect_y0_slider]),
    HBox([rect_x1_slider, rect_y1_slider]),
    polygon_text,
    show_binary_checkbox,
    show_processed_checkbox,
    run_button,
])

display(HBox([controls_left, controls_right]))
display(output)

_run_preview()


Output()

In [18]:

def current_parameter_dict():
    return {
        "diff_video_path": diff_video_path,
        "frame_idx": frame_slider.value,
        "ball_diameter_px": ball_diameter_slider.value,
        "diff_thresh": diff_thresh_slider.value,
        "min_area_scale": min_area_scale_slider.value,
        "max_area_scale": max_area_scale_slider.value,
        "min_bbox_scale": min_bbox_scale_slider.value,
        "max_bbox_scale": max_bbox_scale_slider.value,
        "min_circularity": min_circularity_slider.value,
        "min_mean_intensity": min_mean_intensity_slider.value,
        "max_aspect_ratio": max_aspect_ratio_slider.value,
        "top_k": top_k_slider.value,
        "gaussian_blur_ksize": gaussian_blur_slider.value,
        "region_mode": region_mode_dropdown.value,
        "rect": (
            rect_x0_slider.value,
            rect_y0_slider.value,
            rect_x1_slider.value,
            rect_y1_slider.value,
        ),
        "polygon_points": parse_polygon_text(polygon_text.value),
        "show_binary": show_binary_checkbox.value,
        "show_processed": show_processed_checkbox.value,
    }

current_parameter_dict()


{'diff_video_path': 'diff videos/adjacent_frame_difference.mp4',
 'frame_idx': 0,
 'ball_diameter_px': 10.0,
 'diff_thresh': 22,
 'min_area_scale': 0.2,
 'max_area_scale': 2.5,
 'min_bbox_scale': 0.5,
 'max_bbox_scale': 1.8,
 'min_circularity': 0.35,
 'min_mean_intensity': 15.0,
 'max_aspect_ratio': 2.2,
 'top_k': 5,
 'gaussian_blur_ksize': 3,
 'region_mode': 'none',
 'rect': (0, 0, 400, 400),
 'polygon_points': [(100, 100), (500, 100), (550, 350), (80, 350)],
 'show_binary': True,
 'show_processed': False}